# Step 0 — Applicant Harmonization Using Rosa's Extraction (TEST)

**Purpose:** Reuse Rosa's `tls206_207_persons.parquet` rather than re-query PATSTAT or rerun Riky's keyword/SBERT pipeline. This is an adaptation of Soroush's existing harmonization notebook; the name-cleaning and alias-review logic is unchanged.

**Input:** `Step0/outputs/rosa_test/tls206_207_persons.parquet` exported by Rosa's `Rosa-3-DataExtraction.ipynb`. Optionally place `families.parquet` beside it for an input-population cross-check. Input must come from the **same PATSTAT environment (TEST)** and the same provisional Rosa selection.

**Outputs:** `Step0/outputs/applicant_harmonization_rosa_test/` (a separate folder, so previous harmonization outputs are not overwritten).

**Important:** Rosa's applicant file contains both inventors and applicants. We retain only `is_applicant == True`; we rename her country column `ctry` to `person_ctry_code` to match this notebook. Family selection is provisional until the team agrees on the filters. Never auto-merge proposed aliases; inspect and approve them manually.


## 1. Locate Rosa's locally extracted TEST files

Run Rosa's extraction notebook first and save its outputs to `Step0/outputs/rosa_test/` in this repository. This cell reads those locally generated Parquet files; you do **not** need to create a separate `data/` directory or re-query PATSTAT here. All applicant-harmonization outputs go to a distinct folder so earlier experiments are preserved.

In [2]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
from IPython.display import display

# Project directory
HERE = Path.home() / "flying-drones-patent-analytics" / "Step0"

ENV = "PROD"

# Rosa's official extraction folder
OFFICIAL_DIR = Path("/home/jovyan/progetto/extract") / ENV.lower()

# Our previous TEST extraction folder
TEST_DIR = HERE / "outputs" / "rosa_test"

if (OFFICIAL_DIR / "tls206_207_persons.parquet").is_file():
    DATA_DIR = OFFICIAL_DIR
    print("Using Rosa's official extraction")

elif ENV == "TEST" and (TEST_DIR / "tls206_207_persons.parquet").is_file():
    DATA_DIR = TEST_DIR
    print("Using previous TEST extraction (provisional)")

else:
    raise FileNotFoundError(
        "Extraction files not found. Run Rosa's extraction notebook first."
    )

PERSONS_FILE = DATA_DIR / "tls206_207_persons.parquet"
FAMILIES_FILE = DATA_DIR / "families.parquet"

# Separate harmonization output
OUT = HERE / "outputs" / "applicant_harmonization_test"
OUT.mkdir(parents=True, exist_ok=True)

print("Applicant input:", PERSONS_FILE)
print("Harmonization output:", OUT)

if not PERSONS_FILE.is_file():
    raise FileNotFoundError(f"Missing {PERSONS_FILE}")

Using Rosa's official extraction
Applicant input: /home/jovyan/progetto/extract/prod/tls206_207_persons.parquet
Harmonization output: /home/s353617/flying-drones-patent-analytics/Step0/outputs/applicant_harmonization_test


## 2. Load Rosa's extracted applicants (no PATSTAT query)

Rosa's `tls206_207_persons.parquet` includes inventor-only rows as well as applicant rows. Select only applicants, map Rosa's `ctry` to the `person_ctry_code` used in our existing harmonization logic, and verify unique application–person relationships. Optionally confirm that the applicant families belong to Rosa's extracted `families.parquet`.


In [3]:
persons = pd.read_parquet(PERSONS_FILE)
required = {
    "docdb_family_id", "appln_id", "person_id", "person_name",
    "psn_name", "ctry", "is_applicant"
}
missing_cols = required - set(persons.columns)
if missing_cols:
    raise ValueError(f"Rosa file is missing columns: {sorted(missing_cols)}")

# Rosa's extraction includes both inventors and applicants.
df_applicants = persons.loc[persons["is_applicant"].eq(True)].copy()
df_applicants = df_applicants.rename(columns={"ctry": "person_ctry_code"})
df_applicants = df_applicants.drop_duplicates(
    subset=["docdb_family_id", "appln_id", "person_id"]
).copy()

if df_applicants.empty:
    raise ValueError("No applicant records after filtering is_applicant == True")

print("Raw Rosa person rows:", len(persons))
print("Applicant relationships:", len(df_applicants))
print("Distinct applicant IDs:", df_applicants["person_id"].nunique())
print("Distinct families with applicants:", df_applicants["docdb_family_id"].nunique())

if FAMILIES_FILE.exists():
    rosa_families = pd.read_parquet(FAMILIES_FILE)
    if "docdb_family_id" not in rosa_families.columns:
        raise ValueError("Rosa's families.parquet lacks docdb_family_id")
    family_ids = set(rosa_families["docdb_family_id"].dropna().astype("int64"))
    applicant_families = set(df_applicants["docdb_family_id"].dropna().astype("int64"))
    unexpected = applicant_families - family_ids
    if unexpected:
        raise ValueError(f"Applicant records contain {len(unexpected)} families outside Rosa's list")
    print("Families in Rosa's extracted selection:", len(family_ids))
    print("Families with no applicant rows:", len(family_ids - applicant_families))
else:
    print("Optional cross-check skipped: families.parquet not provided")

display(df_applicants.head())
df_applicants.to_csv(OUT / "applicants_raw.csv", index=False)


Raw Rosa person rows: 708230
Applicant relationships: 169122
Distinct applicant IDs: 53000
Distinct families with applicants: 106877
Families in Rosa's extracted selection: 106998
Families with no applicant rows: 121


,appln_id,person_id,applt_seq_nr,invt_seq_nr,person_name,psn_id,psn_name,psn_sector,han_id,han_name,person_ctry_code,person_address,nuts,nuts_level,docdb_family_id,is_applicant,is_inventor
0,56844380,18189675,1,0,"INSTITUTE OF AUTOMATION, CHINESE ACADEMY OF SC...",4929132,CHINESE ACADEMY OF SCIENCES,GOV NON-PROFIT UNIVERSITY,264,SHANGHAI INSTITUTE OF MATERIA MEDICA CHINESE A...,CN,,,9,40462391,True,False
1,56818693,5401641,7,0,The United States of America as represented by...,23208428,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,GOV NON-PROFIT,1565,THE UNITED STATES OF AMERICA AS REPRESENTED BY...,US,"Washington,DC",,9,40453413,True,False
2,334162736,7806661,9,0,The United States of America as represented by...,23208428,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,GOV NON-PROFIT,1565,THE UNITED STATES OF AMERICA AS REPRESENTED BY...,US,"Washington,DC",,9,40453413,True,False
3,45717965,37152679,1,0,"UNITED STATES OF AMERICA, AS REPRESENTED BY TH...",23208428,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,GOV NON-PROFIT,1565,THE UNITED STATES OF AMERICA AS REPRESENTED BY...,US,,,9,33513736,True,False
4,338939676,11092649,3,0,"The United States of America, as represented b...",23208428,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,GOV NON-PROFIT,1565,THE UNITED STATES OF AMERICA AS REPRESENTED BY...,US,"Washington,DC",,9,44910892,True,False


## 3. Conservative name harmonization

Normalize case/Unicode/spacing and prefer `psn_name` over `person_name`. For entity IDs, prefer **positive `psn_id`**, if available; otherwise use the original `person_id`. This is intentionally conservative: it avoids automatically merging two distinct entities simply because their spellings are similar.

The **HAN fields are retained for later corporate-group mapping**, not automatically used to collapse distinct legal entities.

In [4]:
%pip install cleanco


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import re
import pandas as pd
from IPython.display import display
from cleanco import basename

# Choose the method used for the final display name:
# "psn", "dictionary", or "cleanco"
NAME_METHOD = "psn"

# Professor's legal-form dictionary
LEGAL_FORMS = {
    "INC", "INCORPORATED", "CORP", "CORPORATION",
    "CO", "COMPANY", "LTD", "LIMITED", "LLC",
    "LP", "PLC", "GMBH", "AG", "KG", "MBH",
    "SE", "SA", "SAS", "SPA", "SRL", "BV",
    "NV", "AB", "AS", "ASA", "OY", "OYJ",
    "KK", "KABUSHIKI", "KAISHA", "LTDA",
    "PTY", "THE", "AKTIENGESELLSCHAFT",
    "GESELLSCHAFT"
}

# Basic normalization
def clean_names(series):
    return (
        series.astype("string")
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )

# METHOD A — Professor's Dictionary Cleaning
def clean_with_dictionary(name):
    if pd.isna(name) or not str(name).strip():
        return pd.NA

    words = re.sub(
        r"[^A-Z0-9 ]", " ", str(name).upper()
    ).split()

    while words and words[-1] in LEGAL_FORMS:
        words.pop()

    while words and words[0] in LEGAL_FORMS:
        words.pop(0)

    return " ".join(words) if words else pd.NA

# METHOD B — Professor's Cleanco
def clean_with_cleanco(name):
    if pd.isna(name) or not str(name).strip():
        return pd.NA

    return clean_with_dictionary(
        basename(str(name))
    )

# METHOD C — PATSTAT PSN baseline
# Use our actual Flying Drones applicants
h = df_applicants.copy()

h["person_name_clean"] = clean_names(
    h["person_name"]
)
h["psn_name_clean"] = clean_names(
    h["psn_name"]
)

base_names = h["psn_name_clean"].fillna(
    h["person_name_clean"]
)

h["name_source"] = "person_name"
h.loc[
    h["psn_name_clean"].notna(),
    "name_source"
] = "psn_name"

# Calculate all three methods
h["candidate_psn"] = base_names

h["candidate_dictionary"] = (
    base_names.map(clean_with_dictionary)
    .astype("string")
)

h["candidate_cleanco"] = (
    base_names.map(clean_with_cleanco)
    .astype("string")
)

methods = {
    "psn": "candidate_psn",
    "dictionary": "candidate_dictionary",
    "cleanco": "candidate_cleanco"
}

h["standardized_name"] = (
    h[methods[NAME_METHOD]].fillna(base_names)
)

# Keep PATSTAT identity identifiers conservative.
# Other name-cleaning methods are candidates for review,
# NOT automatically confirmed corporate identities.
person_ids = pd.to_numeric(
    h["person_id"], errors="coerce"
)

if person_ids.isna().any():
    raise ValueError("Missing person_id")

h["entity_id"] = (
    "PERSON:" +
    person_ids.astype("int64").astype(str)
)
h["identity_rule"] = "original_person_id"

if "psn_id" in h.columns:
    psn_ids = pd.to_numeric(
        h["psn_id"], errors="coerce"
    )
    valid = psn_ids.gt(0)

    h.loc[valid, "entity_id"] = (
        "PSN:" +
        psn_ids.loc[valid].astype("int64").astype(str)
    )
    h.loc[valid, "identity_rule"] = "patstat_psn_id"

# Data integrity checks
assert len(h) == len(df_applicants)
assert not h[
    ["docdb_family_id", "appln_id", "person_id"]
].duplicated().any()

# Compare methods
comparison = pd.DataFrame({
    "Method": [
        "Original person names",
        "PATSTAT PSN",
        "Dictionary",
        "Cleanco",
        "Conservative entity IDs"
    ],
    "Unique values": [
        h["person_name_clean"].nunique(),
        h["candidate_psn"].nunique(),
        h["candidate_dictionary"].nunique(),
        h["candidate_cleanco"].nunique(),
        h["entity_id"].nunique()
    ]
})

print("APPLICATION–APPLICANT RECORDS:", len(h))
print("SELECTED METHOD:", NAME_METHOD)
print("\nHARMONIZATION COMPARISON")
display(comparison)

# Inspect the actual transformations
display(
    h[[
        "person_name",
        "psn_name",
        "candidate_dictionary",
        "candidate_cleanco",
        "standardized_name",
        "entity_id"
    ]].head(20)
)

# Potential alias groups: different PATSTAT entities
# that become the same dictionary-cleaned name.
alias_candidates = (
    h.dropna(subset=["candidate_dictionary"])
    .groupby(
        ["candidate_dictionary", "person_ctry_code"],
        dropna=False
    )
    .agg(
        distinct_entities=("entity_id", "nunique"),
        example_names=(
            "person_name",
            lambda s: " | ".join(
                sorted(set(s.dropna().astype(str)))[:5]
            )
        )
    )
    .reset_index()
)

alias_candidates = alias_candidates[
    alias_candidates["distinct_entities"] > 1
].sort_values(
    "distinct_entities", ascending=False
)

print("\nPOTENTIAL ALIAS GROUPS:", len(alias_candidates))
display(alias_candidates.head(20))

alias_candidates.to_csv(
    OUT / "dictionary_alias_candidates.csv",
    index=False
)


APPLICATION–APPLICANT RECORDS: 169122
SELECTED METHOD: psn

HARMONIZATION COMPARISON


,Method,Unique values
0,Original person names,42873
1,PATSTAT PSN,40821
2,Dictionary,39138
3,Cleanco,39079
4,Conservative entity IDs,42769


,person_name,psn_name,candidate_dictionary,candidate_cleanco,standardized_name,entity_id
0,"INSTITUTE OF AUTOMATION, CHINESE ACADEMY OF SC...",CHINESE ACADEMY OF SCIENCES,CHINESE ACADEMY OF SCIENCES,CHINESE ACADEMY OF SCIENCES,CHINESE ACADEMY OF SCIENCES,PSN:4929132
1,The United States of America as represented by...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,PSN:23208428
2,The United States of America as represented by...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,PSN:23208428
3,"UNITED STATES OF AMERICA, AS REPRESENTED BY TH...",NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,PSN:23208428
4,"The United States of America, as represented b...",NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,PSN:23208428
5,The United States of America as represented by...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,PSN:23208428
6,The United States of America as represented by...,THE UNITED STATES OF AMERICA AS REPRESENTED BY...,UNITED STATES OF AMERICA AS REPRESENTED BY THE...,UNITED STATES OF AMERICA AS REPRESENTED BY THE...,THE UNITED STATES OF AMERICA AS REPRESENTED BY...,PSN:32464529
7,The United States of America as represented by...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA NATIONAL AERONAUTICS AND SPACE ADMINISTRA...,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTR...,PSN:23208428
8,21ST CENTURY AIRSHIPS INC.,21ST CENTURY AIRSHIPS,21ST CENTURY AIRSHIPS,21ST CENTURY AIRSHIPS,21ST CENTURY AIRSHIPS,PSN:46837
9,21st Century Airships Inc.,21ST CENTURY AIRSHIPS,21ST CENTURY AIRSHIPS,21ST CENTURY AIRSHIPS,21ST CENTURY AIRSHIPS,PSN:46837



POTENTIAL ALIAS GROUPS: 1889


,candidate_dictionary,person_ctry_code,distinct_entities,example_names
7129,CN,,31,中国人民公安大学（CN） | 中国民航大学（CN） | 中国科学院信息工程研究所（CN） |...
1159,ANDURIL INDUSTRIES,US,15,ANDURIL INDUSTRIES INC | ANDURIL INDUSTRIES IN...
31677,SKYDIO,US,11,"SKYDIO, INC. | Skydio Inc. | Skydio, Inc | Sky..."
472,AEROVIRONMENT,US,11,"AERO VIRONMENT, INC. | AEROVIRONMENT | AEROVIR..."
1947,AUTOFLIGHT KUNSHAN,CN,9,"AUTOFLIGHT (KUNSHAN) CO., LTD | AUTOFLIGHT (KU..."
26787,RAYTHEON,US,9,RAYTHEON COMPANY | Raytheon Company | Raytheon...
24773,O TOOLE DANIEL S,US,9,O'TOOLE DANIEL S. | O'TOOLE Daniel S. | O'TOOL...
1677,ARCHER AVIATION,US,9,"ARCHER AVIATION INC. | ARCHER AVIATION, INC. |..."
33977,TEXTRON SYSTEMS,US,8,TEXTRON SYSTEMS CORPORATION | Textron Systems ...
1937,AUTEL ROBOTICS,CN,8,"AUTEL ROBOTICS CO LTD | AUTEL ROBOTICS CO., LT..."


## 4. Flag possible aliases for manual review

Two different entity IDs with the same normalized name and country are **candidates**, not proof of identity. They are exported for review but are **not merged automatically**. This is where you can add carefully verified aliases and documented subsidiary rules as a later improvement.

In [6]:

# STEP 0 — CELL 4: FLAG POSSIBLE APPLICANT ALIASES
# Based on the professor's name harmonization methods.
# Potential matches are flagged, not automatically merged.

import pandas as pd
from IPython.display import display

# Preserve the original PSN-based key for compatibility
# with the following export cell.
h["name_country_key"] = (
    h["standardized_name"].fillna("").astype(str)
    + " | "
    + h["person_ctry_code"].fillna("").astype(str).str.upper()
)

def flag_possible_aliases(df, name_column):
    """
    Flag records where the same cleaned applicant name
    and country correspond to different entity IDs.

    This is evidence of a possible alias, not proof
    that the organizations are identical.
    """
    names = df[name_column].astype("string").str.strip()

    countries = (
        df["person_ctry_code"]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
    )

    temp = pd.DataFrame({
        "name": names,
        "country": countries,
        "entity_id": df["entity_id"]
    }, index=df.index)

    valid = temp["name"].notna() & temp["name"].ne("")

    distinct_ids = (
        temp.loc[valid]
        .groupby(["name", "country"], dropna=False)["entity_id"]
        .transform("nunique")
    )

    flags = pd.Series(False, index=df.index)
    flags.loc[valid] = distinct_ids.gt(1).to_numpy()

    return flags


# Compare possible aliases under all three methods.
h["flag_psn"] = flag_possible_aliases(
    h, "candidate_psn"
)

h["flag_dictionary"] = flag_possible_aliases(
    h, "candidate_dictionary"
)

h["flag_cleanco"] = flag_possible_aliases(
    h, "candidate_cleanco"
)

h["flag_missing_name"] = h["standardized_name"].isna()

# A record needs review if any method raises a flag.
flag_columns = [
    "flag_psn",
    "flag_dictionary",
    "flag_cleanco",
    "flag_missing_name"
]

h["needs_manual_review"] = h[flag_columns].any(axis=1)

# Create a separate manual-review dataset.
review_columns = [
    "docdb_family_id",
    "appln_id",
    "person_id",
    "person_name",
    "psn_name",
    "candidate_dictionary",
    "candidate_cleanco",
    "standardized_name",
    "person_ctry_code",
    "entity_id",
    "identity_rule",
    "name_country_key"
] + flag_columns

review = h.loc[
    h["needs_manual_review"], review_columns
].copy()

review["human_decision"] = ""
review["notes"] = ""

# Summarize the results.
summary_flags = pd.DataFrame({
    "Review condition": [
        "PSN name collision",
        "Dictionary candidate collision",
        "Cleanco candidate collision",
        "Missing standardized name",
        "Total records requiring review"
    ],
    "Number of records": [
        int(h["flag_psn"].sum()),
        int(h["flag_dictionary"].sum()),
        int(h["flag_cleanco"].sum()),
        int(h["flag_missing_name"].sum()),
        len(review)
    ]
})

print("Total applicant records:", len(h))
print("Records requiring manual review:", len(review))
print("Distinct entity IDs flagged:",
      review["entity_id"].nunique())

display(summary_flags)

print("\nEXAMPLES REQUIRING REVIEW")
display(review.head(20))


Total applicant records: 169122
Records requiring manual review: 37063
Distinct entity IDs flagged: 4829


,Review condition,Number of records
0,PSN name collision,12579
1,Dictionary candidate collision,36676
2,Cleanco candidate collision,37002
3,Missing standardized name,0
4,Total records requiring review,37063



EXAMPLES REQUIRING REVIEW


,docdb_family_id,appln_id,person_id,person_name,psn_name,candidate_dictionary,candidate_cleanco,standardized_name,person_ctry_code,entity_id,identity_rule,name_country_key,flag_psn,flag_dictionary,flag_cleanco,flag_missing_name,human_decision,notes
53,42107874,471289229,54298443,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
54,34272380,51442264,5504730,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
55,36992248,49635562,5504730,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
56,42107874,315441329,5255120,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
57,39491376,56262965,4634870,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
58,39491376,335577032,5504730,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
59,34272380,323373124,5255120,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
60,45095447,409765155,5255120,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
61,34272380,2696952,5255120,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,
62,34272380,53943191,5504730,AAI Corporation,AAI CORPORATION,AAI,AAI,AAI CORPORATION,US,PSN:158446,patstat_psn_id,AAI CORPORATION | US,True,True,True,False,,


In [7]:
# One row per distinct entity within each Cleanco candidate group
unique_review = (
    h.loc[
        h["needs_manual_review"],
        [
            "candidate_cleanco",
            "person_ctry_code",
            "entity_id",
            "psn_name",
            "person_name",
            "han_name"
        ]
    ]
    .drop_duplicates(
        subset=[
            "candidate_cleanco",
            "person_ctry_code",
            "entity_id"
        ]
    )
    .sort_values([
        "candidate_cleanco",
        "person_ctry_code",
        "entity_id"
    ])
    .reset_index(drop=True)
)

print("Unique entity-level review entries:", len(unique_review))
display(unique_review.head(40))

Unique entity-level review entries: 4901


,candidate_cleanco,person_ctry_code,entity_id,psn_name,person_name,han_name
0,10112755 CANADA,CA,PSN:185987104,10112755 Canada Inc.,10112755 Canada Inc.,10112755 CA INC
1,10112755 CANADA,CA,PSN:38457,10112755 CANADA,10112755 CANADA INC.,10112755 CA INC
2,10757969 CANADA,CA,PSN:181601304,10757969 Canada Corporation,10757969 Canada Corporation,10757969 CA CORP
3,10757969 CANADA,CA,PSN:38613,10757969 CANADA CORPORATION,10757969 CANADA CORPORATION,10757969 CA CORP
4,1AHEAD TECHNOLOGIES,US,PSN:185429508,1AHEAD TECHNOLOGIES,1AHEAD TECHNOLOGIES,1AHEAD TECH
5,1AHEAD TECHNOLOGIES,US,PSN:40575,1AHEAD TECHNOLOGIES,1AHEAD Technologies,1AHEAD TECH
6,36TH RESEARCH INSTITUTE OF CETC,,PSN:32406926,THE 36TH RESEARCH INSTITUTE OF CETC,THE 36TH RESEARCH INSTITUTE OF CETC,THE 36TH RESEARCH INSTITUTE OF CETC
7,36TH RESEARCH INSTITUTE OF CETC,,PSN:53532,36TH RESEARCH INSTITUTE OF CETC,36th Research Institute of CETC,36th Research Institute of CETC
8,4596286 MANITOBA,CA,PSN:185986976,4596286 Manitoba Ltd,4596286 Manitoba Ltd,4596286 MANITOBA LTD
9,4596286 MANITOBA,CA,PSN:189029624,4596286 Manitoba Inc.,4596286 Manitoba Inc.,4596286 MANITOBA INC


In [8]:

# STEP 0 — APPLICANT HARMONIZATION
# Cell 4B: Prepare alias groups for manual review
#
# Based on professor's Dictionary / Cleanco methodology.
# This cell DOES NOT merge applicants or modify PATSTAT IDs.

from pathlib import Path
import pandas as pd
from IPython.display import display

OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)

# 1. Prepare the existing harmonized applicant records
candidates = h.copy()

candidates["_candidate_name"] = (
    candidates["candidate_cleanco"]
    .astype("string")
    .str.strip()
)

candidates["_country"] = (
    candidates["person_ctry_code"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

candidates["_entity"] = (
    candidates["entity_id"]
    .astype("string")
    .str.strip()
)

candidates = candidates.dropna(
    subset=["_candidate_name", "_entity"]
)

candidates = candidates[
    candidates["_candidate_name"].ne("")
    & candidates["_entity"].ne("")
].copy()


# 2. Helper: collect unique names without repeating them
def unique_values(series, limit=15):
    values = sorted({
        str(value).strip()
        for value in series.dropna()
        if str(value).strip()
    })
    result = " | ".join(values[:limit])

    if len(values) > limit:
        result += f" | ... (+{len(values) - limit} more)"

    return result


# 3. Build one review row per candidate group
review_rows = []

for (name, country), group in candidates.groupby(
    ["_candidate_name", "_country"],
    sort=True
):
    entity_ids = sorted(group["_entity"].unique())

    # Only review groups containing different identities
    if len(entity_ids) < 2:
        continue

    # Details for every entity in the group
    entity_details = []

    for entity_id, entity_rows in group.groupby("_entity"):
        families = entity_rows["docdb_family_id"].nunique()
        original_names = unique_values(
            entity_rows["person_name"], limit=3
        )

        entity_details.append(
            f"{entity_id} ({families} families): "
            f"{original_names}"
        )

    review_rows.append({
        "candidate_name": name,
        "country": country,
        "n_entities": len(entity_ids),
        "n_families": group["docdb_family_id"].nunique(),
        "n_records": len(group),
        "entity_ids": " | ".join(entity_ids),
        "psn_names": unique_values(group["psn_name"]),
        "original_names": unique_values(
            group["person_name"]
        ),
        "han_names": (
            unique_values(group["han_name"])
            if "han_name" in group.columns else ""
        ),
        "han_ids": (
            unique_values(group["han_id"])
            if "han_id" in group.columns else ""
        ),
        "entity_details": " ; ".join(entity_details),

        # To be completed after reviewing the evidence
        "decision": "",
        "canonical_entity_id": "",
        "review_notes": ""
    })


# 4. Prioritize groups affecting more patent families
alias_review_groups = pd.DataFrame(review_rows)

if not alias_review_groups.empty:
    alias_review_groups = (
        alias_review_groups
        .sort_values(
            ["n_families", "n_entities", "candidate_name"],
            ascending=[False, False, True]
        )
        .reset_index(drop=True)
    )

    alias_review_groups.insert(
        0,
        "group_id",
        range(1, len(alias_review_groups) + 1)
    )

# 5. Export review table
output_file = OUT / "alias_review_groups.csv"

alias_review_groups.to_csv(
    output_file,
    index=False
)

print("APPLICANT ALIAS REVIEW")
print("----------------------")
print("Candidate groups:", len(alias_review_groups))

if not alias_review_groups.empty:
    print(
        "Distinct entity IDs:",
        len({
            entity
            for value in alias_review_groups["entity_ids"]
            for entity in value.split(" | ")
        })
    )

    print(
        "Patent families affected:",
        candidates.loc[
            candidates.set_index(
                ["_candidate_name", "_country"]
            ).index.isin(
                alias_review_groups.set_index(
                    ["candidate_name", "country"]
                ).index
            ),
            "docdb_family_id"
        ].nunique()
    )

print("\nSaved to:", output_file.resolve())

display(
    alias_review_groups[
        [
            "group_id",
            "candidate_name",
            "country",
            "n_entities",
            "n_families",
            "entity_ids",
            "psn_names",
            "han_names",
            "decision"
        ]
    ].head(30)
)


APPLICANT ALIAS REVIEW
----------------------
Candidate groups: 1918
Distinct entity IDs: 4819
Patent families affected: 18510

Saved to: /home/jovyan/flying-drones-patent-analytics/Step0/outputs/applicant_harmonization_test/alias_review_groups.csv


,group_id,candidate_name,country,n_entities,n_families,entity_ids,psn_names,han_names,decision
0,1,SZ DJI TECHNOLOGY,CN,4,1879,PSN:183472139 | PSN:188916425 | PSN:199369275 ...,"SZ DJI TECHNOLOGY CO., LTD. | SZ DJI TECHNOLOG...",SHENZHEN DAJIANG INNOVATION TECH CO LTD | SZ D...,
1,2,SZ DJI TECHNOLOGY,,2,1520,PSN:184518079 | PSN:31716938,SZ DJI TECHNOLOGY CO LTD | SZ DJI TECHNOLOGY C...,DA-JIANG INNOVATIONS SCIENCE AND TECHNOLOGY CO...,
2,3,NANJING UNIVERSITY OF AERONAUTICS AND ASTRONAU...,,2,1242,PSN:190389704 | PSN:23153515,NANJING UNIVERSITY OF AERONAUTICS AND ASTRONAU...,NANJING UNIV OF AERONAUTICS AND ASTRONAUTICS |...,
3,4,NORTHWESTERN POLYTECHNICAL UNIVERSITY,,2,1031,PSN:189586574 | PSN:23854997,NORTHWESTERN POLYTECHNICAL UNIVERSITY,NORTHWESTERN POLYTECHNICAL UNIVERSITY,
4,5,GUANGDONG POWER GRID,,2,731,PSN:11095093 | PSN:190441444,"GUANGDONG POWER GRID CO., LTD. | GUANGDONG POW...",DONGGUAN POWER BUREAU OF GUANGDONG POWER GRID ...,
5,6,AUTEL ROBOTICS,CN,8,522,PSN:1626020 | PSN:183423209 | PSN:190486986 | ...,"AUTEL ROBOTICS CO., LTD | AUTEL ROBOTICS CO., ...",AUTEL ROBOTICS CO LTD | AUTEL ROBOTICS LTD,
6,7,SHENZHEN AUTEL INTELLIGENT AVIATION TECHNOLOGY,,2,396,PSN:182897208 | PSN:29587963,SHENZHEN AUTEL INTELLIGENT AVIATION TECHNOLOGY...,SHENZHEN AUTEL INTELLIGENT AVIATION TECHNOLOGY...,
7,8,ZHEJIANG UNIVERSITY,,2,382,PSN:37204058 | PSN:42952880,ZHEJIANG UNIVERSITY | 浙江大学ＺＨＥＪＩＡＮＧ ＵＮＩＶＥＲＳＩＴＹ,BINHAI INDUSTRIAL TECHNOLOGY RESEARCH INSTITUT...,
8,9,BOEING,US,5,377,PSN:182885097 | PSN:189008331 | PSN:189177142 ...,BOEING COMPANY | THE BOEING COMPANY | The Boei...,TE BOING KOMPANI | THE BOEING CO | THE BOEING ...,
9,10,XAIRCRAFT,,2,372,PSN:187880229 | PSN:35706380,"XAIRCRAFT CO., LTD. | XAIRCRAFT COMPANY","XAIRCRAFT CO., LTD.",


## 5. Export the four deliverables and quality checks

Save a traceable mapping and a **deduplicated family-to-applicant link table** for downstream competitive-landscape analysis. Counting applications or CSV rows as patents would overcount the inventions; use `docdb_family_id` at the family level.

In [9]:
h.drop(columns=["name_country_key"]).to_csv(OUT / "applicants_harmonized.csv", index=False)

mapping_cols = [c for c in [
    "person_id", "person_name", "psn_name", "psn_id", "han_name", "han_id",
    "person_ctry_code", "standardized_name", "entity_id", "identity_rule",
    "needs_manual_review"
] if c in h.columns]
entity_mapping = h[mapping_cols].drop_duplicates().sort_values("entity_id")
entity_mapping.to_csv(OUT / "applicant_entity_mapping.csv", index=False)

family_links = h[["docdb_family_id", "entity_id"]].drop_duplicates()
family_links.to_csv(OUT / "family_applicant_links.csv", index=False)
review.to_csv(OUT / "applicant_manual_review.csv", index=False)

summary = pd.DataFrame([{
    "applicant_records":len(h),
    "families_with_applicants":h.docdb_family_id.nunique(),
    "distinct_person_ids":h.person_id.nunique(),
    "harmonized_entity_ids":h.entity_id.nunique(),
    "manual_review_records":len(review),
}])
summary.to_csv(OUT / "summary.csv", index=False)
display(summary)
print("Outputs saved to:", OUT.resolve())


,applicant_records,families_with_applicants,distinct_person_ids,harmonized_entity_ids,manual_review_records
0,169122,106877,53000,42769,37063


Outputs saved to: /home/jovyan/flying-drones-patent-analytics/Step0/outputs/applicant_harmonization_test


### How to interpret this run

This notebook uses Rosa's **post-year-filter and PI-only provisional dataset**, so its family and applicant counts may differ from the earlier harmonization experiment based on Riky's broader selection.

The PSN, dictionary, and Cleanco comparisons remain the same. The alias-review output generated in this run is a **new candidate list**, not the previously reviewed 29 groups. Review any groups manually before merging entities. Re-run against the final approved family population once validation and filter decisions are complete.
